# Validación de `src/estimacion.py` — Etapa 2

Reutiliza los datos ya cacheados en la Etapa 1 (mismo recorte: ventana 2025-03-11 → 2025-09-18, columnas con >10% de faltantes eliminadas, imputación aplicada) para validar los estimadores de $\mu$ y $\Sigma$.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.covariance import LedoitWolf

from src.config import cargar_config
from src import datos
from src import estimacion

config = cargar_config()
factor_anual = config["estimacion"]["factor_anualizacion"]

## Preparación de datos (misma ventana validada en la Etapa 1)

In [2]:
precios = datos.obtener_precios(config)
benchmark = datos.obtener_benchmark(config)

periodo_inicio = "2025-03-11"
periodo_fin = "2025-09-18"

ventana = precios.loc[periodo_inicio:periodo_fin]
ventana_filtrada = datos.eliminar_columnas_incompletas(ventana, umbral=0.10)
ventana_imputada = datos.imputar_precios(ventana_filtrada)

rendimientos_log = datos.calcular_rendimientos_log(ventana_imputada)

benchmark_ventana = benchmark.loc[periodo_inicio:periodo_fin]
rendimientos_benchmark = datos.calcular_rendimientos_log(benchmark_ventana.to_frame(name=config["benchmark"])).iloc[:, 0]

assert rendimientos_benchmark.index.equals(rendimientos_log.index)

print(f"Shape rendimientos activos: {rendimientos_log.shape}")
print(f"NaN en rendimientos benchmark: {rendimientos_benchmark.isna().sum()}")

Shape rendimientos activos: (132, 110)
NaN en rendimientos benchmark: 0


## $\mu$ histórico

In [3]:
mu_historico = estimacion.estimar_mu_historico(rendimientos_log, factor_anual=factor_anual)
mu_historico_manual = rendimientos_log.mean() * factor_anual

assert mu_historico.equals(mu_historico_manual)

display(mu_historico.describe())

count    110.000000
mean       0.185249
std        0.447450
min       -1.576324
25%        0.000000
50%        0.122054
75%        0.390050
max        1.984740
dtype: float64

## $\mu$ Bayes-Stein (Jorion)

El shrinkage debe reducir la dispersión entre activos frente a la media muestral pura, y λ debe caer entre 0 y 1.

In [4]:
mu_bayes_stein = estimacion.estimar_mu_bayes_stein(rendimientos_log, factor_anual=factor_anual)

print(f"Desv. estandar mu historico: {mu_historico.std():.4f}")
print(f"Desv. estandar mu Bayes-Stein: {mu_bayes_stein.std():.4f}")

assert mu_bayes_stein.std() < mu_historico.std()

display(mu_bayes_stein.describe())

Desv. estandar mu historico: 0.4475
Desv. estandar mu Bayes-Stein: 0.3096


count    110.000000
mean       0.130236
std        0.309641
min       -1.088792
25%        0.002042
50%        0.086505
75%        0.271961
max        1.375504
dtype: float64

## Regresión CAPM vectorizada

Tasa libre de riesgo ilustrativa (aprox. CETES 28 días anualizada) solo para esta validación; en la interfaz será un parámetro que el usuario elija.

Cruzamos el resultado vectorizado contra `statsmodels.OLS` corrido individualmente sobre algunos activos, para confirmar que el álgebra matricial es correcta.

In [5]:
tasa_libre_riesgo_anual = 0.08

resultado_capm = estimacion.regresion_capm(
    rendimientos_log, rendimientos_benchmark,
    tasa_libre_riesgo_anual=tasa_libre_riesgo_anual, factor_anual=factor_anual)

r_f_diario = tasa_libre_riesgo_anual / factor_anual
exceso_mercado = rendimientos_benchmark - r_f_diario

activos_prueba = rendimientos_log.columns[:3]

for ticker in activos_prueba:
    exceso_activo = rendimientos_log[ticker] - r_f_diario
    X_sm = sm.add_constant(exceso_mercado.values)
    modelo_sm = sm.OLS(exceso_activo.values, X_sm).fit()

    alpha_sm, beta_sm = modelo_sm.params
    p_valor_sm = modelo_sm.pvalues[0]

    print(f"{ticker}:")
    print(f"  alpha  vectorizado={resultado_capm['alpha_diario'][ticker]:.6f}  statsmodels={alpha_sm:.6f}")
    print(f"  beta   vectorizado={resultado_capm['beta'][ticker]:.6f}  statsmodels={beta_sm:.6f}")
    print(f"  p_valor vectorizado={resultado_capm['p_valor_alpha'][ticker]:.6f}  statsmodels={p_valor_sm:.6f}")

    assert np.isclose(resultado_capm["alpha_diario"][ticker], alpha_sm)
    assert np.isclose(resultado_capm["beta"][ticker], beta_sm)
    assert np.isclose(resultado_capm["p_valor_alpha"][ticker], p_valor_sm)

AC.MX:
  alpha  vectorizado=-0.001575  statsmodels=-0.001575
  beta   vectorizado=0.713194  statsmodels=0.713194
  p_valor vectorizado=0.201326  statsmodels=0.201326
ALTERNA.MX:
  alpha  vectorizado=0.000094  statsmodels=0.000094
  beta   vectorizado=-0.006408  statsmodels=-0.006408
  p_valor vectorizado=0.087649  statsmodels=0.087649
ASURB.MX:
  alpha  vectorizado=0.000720  statsmodels=0.000720
  beta   vectorizado=0.928235  statsmodels=0.928235
  p_valor vectorizado=0.566833  statsmodels=0.566833


In [6]:
mu_capm = estimacion.estimar_mu_capm(
    rendimientos_log, rendimientos_benchmark,
    tasa_libre_riesgo_anual=tasa_libre_riesgo_anual, factor_anual=factor_anual)

print(f"Prima de mercado anualizada: {resultado_capm['prima_mercado_anual']:.4f}")
display(mu_capm.describe())

Prima de mercado anualizada: 0.2529


count    110.000000
mean       0.175054
std        0.118290
min       -0.000790
25%        0.080000
50%        0.132133
75%        0.265250
max        0.526312
dtype: float64

## $\Sigma$ muestral vs Ledoit-Wolf

In [7]:
sigma_muestral = estimacion.estimar_sigma_muestral(rendimientos_log, factor_anual=factor_anual)
sigma_muestral_manual = rendimientos_log.cov() * factor_anual

assert sigma_muestral.equals(sigma_muestral_manual)

sigma_lw = estimacion.estimar_sigma_ledoit_wolf(rendimientos_log, factor_anual=factor_anual)

modelo_lw_directo = LedoitWolf().fit(rendimientos_log.values)
sigma_lw_manual = modelo_lw_directo.covariance_ * factor_anual

assert np.allclose(sigma_lw.values, sigma_lw_manual)

print(f"Intensidad de shrinkage (Ledoit-Wolf): {modelo_lw_directo.shrinkage_:.4f}")

condicion_muestral = np.linalg.cond(sigma_muestral.values)
condicion_lw = np.linalg.cond(sigma_lw.values)
print(f"Numero de condicion Sigma muestral:    {condicion_muestral:.2e}")
print(f"Numero de condicion Sigma Ledoit-Wolf: {condicion_lw:.2e}")

Intensidad de shrinkage (Ledoit-Wolf): 0.4575
Numero de condicion Sigma muestral:    2.31e+19
Numero de condicion Sigma Ledoit-Wolf: 1.65e+01


## Dispatchers `estimar_mu` / `estimar_sigma`

In [8]:
assert estimacion.estimar_mu(rendimientos_log, metodo="historico", factor_anual=factor_anual).equals(mu_historico)
assert estimacion.estimar_mu(rendimientos_log, metodo="bayes_stein", factor_anual=factor_anual).equals(mu_bayes_stein)
assert estimacion.estimar_mu(
    rendimientos_log, metodo="capm", factor_anual=factor_anual,
    rendimientos_benchmark=rendimientos_benchmark, tasa_libre_riesgo_anual=tasa_libre_riesgo_anual,
).equals(mu_capm)

assert estimacion.estimar_sigma(rendimientos_log, metodo="muestral", factor_anual=factor_anual).equals(sigma_muestral)
assert estimacion.estimar_sigma(rendimientos_log, metodo="ledoit_wolf", factor_anual=factor_anual).equals(sigma_lw)

## Pruebas de hipótesis: FDR (Benjamini-Hochberg) y GRS

In [9]:
reporte_fdr = estimacion.prueba_fdr_alphas(resultado_capm, nivel_significancia=0.05)

n_significativos_sin_correccion = (reporte_fdr["p_valor"] < 0.05).sum()
n_significativos_fdr = reporte_fdr["significativo_fdr"].sum()

print(f"Activos con p_valor < 0.05 sin correccion: {n_significativos_sin_correccion} / {len(reporte_fdr)}")
print(f"Activos significativos tras correccion FDR: {n_significativos_fdr} / {len(reporte_fdr)}")

assert n_significativos_fdr <= n_significativos_sin_correccion

Activos con p_valor < 0.05 sin correccion: 17 / 110
Activos significativos tras correccion FDR: 14 / 110


In [10]:
reporte_grs = estimacion.prueba_grs(
    resultado_capm, rendimientos_benchmark,
    tasa_libre_riesgo_anual=tasa_libre_riesgo_anual, factor_anual=factor_anual)

print(f"Estadistico GRS: {reporte_grs['estadistico_grs']:.4f}")
print(f"P-valor GRS:     {reporte_grs['p_valor_grs']:.4f}")
print(f"Grados de libertad: F({reporte_grs['grados_libertad_num']}, {reporte_grs['grados_libertad_den']})")

if reporte_grs["p_valor_grs"] < 0.05:
    print("\nSe rechaza H0: los alphas conjuntos son significativamente distintos de cero (el CAPM no explica del todo los retornos en esta ventana).")
else:
    print("\nNo se rechaza H0: no hay evidencia conjunta de que los alphas sean distintos de cero.")

Estadistico GRS: 0.6854
P-valor GRS:     0.8925
Grados de libertad: F(110, 21)

No se rechaza H0: no hay evidencia conjunta de que los alphas sean distintos de cero.


Es necesario mencionar: con $N=110$ y sólo 21 grados de libertad en el denominador, la ventana tiene $T = 110 + 21 + 1 = 132$ observaciones. Eso es muy poco margen. Es matemáticamente válido, pero el poder estadístico de la prueba es bajo: con tan pocos grados de libertad relativo al número de activos, cuesta mucho rechazar $H_0$ aunque hubiera alphas reales distintos de cero. Vale la pena tenerlo en mente al interpretar el "no se rechaza" — no es lo mismo que confirmar que el CAPM es correcto, es que no se tiene evidencia suficiente para decir que falla, con el poder disponible.